Focus presentation for decision makers on why customers leave THE Chuurrrrrrnnn column

## Case Study

In [39]:
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.tree import export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBClassifier
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score
from sklearn.metrics import mean_absolute_error
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

In [ ]:
df = pd.read_csv("customer_churn.csv")
df.head() #gender, dependents, partner, OnlineSecurity, DeviceProtection, 

,Unnamed: 0,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,1,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,2,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,3,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,4,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [28]:
df['StreamingTV'].unique()

array(['No', 'Yes', 'No internet service'], dtype=object)

In [30]:
df.iloc[:, 13:]

,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,No,No,One year,No,Mailed check,56.95,1889.5,No
2,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes
...,...,...,...,...,...,...,...,...
7038,Yes,Yes,One year,Yes,Mailed check,84.80,1990.5,No
7039,Yes,Yes,One year,Yes,Credit card (automatic),103.20,7362.9,No
7040,No,No,Month-to-month,Yes,Electronic check,29.60,346.45,No
7041,No,No,Month-to-month,Yes,Mailed check,74.40,306.6,Yes


In [ ]:
df.info() #gender, dependents, partner, OnlineSecurity (Ordinal), Contract(ordinal), DeviceProtection, PhoneService, MultipleLines (ordinal), InternetService (Ordinal), OnlineBackup (Ordinal), 
# DeviceProtection (ordinal), TechSupport(ord), StreamingTV(ord), StreamingMovies(ord), PaperlessBilling (cat), PaymentMethod (cat)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Unnamed: 0        7043 non-null   int64  
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [37]:
df["Contract"].value_counts()


Contract
Month-to-month    3875
Two year          1695
One year          1473
Name: count, dtype: int64

In [16]:
X= df.drop(columns="Churn")
y= df['Churn']

In [ ]:
#gender, dependents, partner, OnlineSecurity (Ordinal), Contract(ordinal), DeviceProtection, PhoneService, MultipleLines (ordinal), InternetService (Ordinal), OnlineBackup (Ordinal), 
# DeviceProtection (ordinal), TechSupport(ord), StreamingTV(ord), StreamingMovies(ord), PaperlessBilling (cat), PaymentMethod (cat)

In [47]:
# make list of cols
ord_cols=['Contract']

cat_cols=df.drop(columns=['Churn','Contract','MonthlyCharges', 'TotalCharges', 'tenure']).columns.tolist()
num_cols=['tenure', 'MonthlyCharges', 'TotalCharges']
cat_cols

['Unnamed: 0',
 'gender',
 'SeniorCitizen',
 'Partner',
 'Dependents',
 'PhoneService',
 'MultipleLines',
 'InternetService',
 'OnlineSecurity',
 'OnlineBackup',
 'DeviceProtection',
 'TechSupport',
 'StreamingTV',
 'StreamingMovies',
 'PaperlessBilling',
 'PaymentMethod']

In [48]:

preprocessor= ColumnTransformer(
    [
        ("ord", OrdinalEncoder(categories=['Month-to-month', 'One year', 'Two year']), ord_cols),
        ("ohe", OneHotEncoder(sparse_output=False, drop='first'), cat_cols),
        ("num", StandardScaler(), num_cols)
    ]
)

In [49]:

from random import Random


pipe= Pipeline([
    ("preprocessor", preprocessor),
    ("RandomForestClassifier", RandomForestClassifier(n_estimators=100, max_depth=7, random_state=42))
])

In [50]:
X_train, X_test, y_train, y_test= train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
    )

In [51]:
pipe.fit(X_train, y_train)

ValueError: Shape mismatch: if categories is an array, it has to be of shape (n_features,).

In [53]:
X_train.shape

(5634, 20)

In [54]:
y_train.shape

(5634,)